# Fund-aware chunking evaluation

Chunking runs **inside the validated structural segments** produced by the previous phase. That makes fund ownership a hard boundary: overlap is allowed within a segment but never across two funds.

The chunker preserves document, fund, page range, section, segment type, and source modality metadata for retrieval and citations.


In [ ]:
from pathlib import Path
import json, sys, importlib
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from fund_rag import chunking
importlib.reload(chunking)

PARSED_DIR = ROOT / "artifacts" / "ingestion"
FUND_DISCOVERY_DIR = ROOT / "artifacts" / "fund_discovery"
SEGMENT_DIR = ROOT / "artifacts" / "fund_segmentation"

documents = [
    json.loads(path.read_text(encoding="utf-8"))
    for path in sorted(PARSED_DIR.glob("*.json"))
]
assert documents, "Run ingestion_eval.ipynb first."
print([d["document"] for d in documents])


## 1. Load upstream artifacts


In [ ]:
registry = json.loads(
    (FUND_DISCOVERY_DIR / "registry.json").read_text(encoding="utf-8")
)
segments = json.loads(
    (SEGMENT_DIR / "segments.json").read_text(encoding="utf-8")
)

print("Registry funds:", len(registry))
print("Segments:", len(segments))


## 2. Use the validated structural segments

This notebook deliberately does **not** call fund discovery or segmentation again; it evaluates only the chunking phase.


In [ ]:
prospectus_segments = [
    segment
    for segment in segments
    if segment["document"] == "FU_BF065_EN.pdf"
]

print("Prospectus segments:", len(prospectus_segments))
pd.DataFrame(prospectus_segments).tail(10)


## 3. Create chunks


In [ ]:
MAX_WORDS = 300
MIN_WORDS = 80
OVERLAP_WORDS = 40

chunks = chunking.chunk_documents(
    documents,
    segments,
    max_words=MAX_WORDS,
    min_words=MIN_WORDS,
    overlap_words=OVERLAP_WORDS,
)

chunk_df = pd.DataFrame(chunks)

print("Total chunks:", len(chunk_df))
display(chunk_df.head())


## 4. Chunk-size distribution

A few short chunks are acceptable around structural boundaries. We mainly want to avoid very large chunks.


In [ ]:
display(
    chunk_df["word_count"]
    .describe(percentiles=[0.1, 0.5, 0.9, 0.95, 0.99])
    .to_frame()
)

print("Chunks above max words:", (chunk_df["word_count"] > MAX_WORDS).sum())
print("Chunks under 50 words:", (chunk_df["word_count"] < 50).sum())


## 5. Fund-boundary safety

Every chunk has exactly one fund value because chunking happens inside one validated segment at a time.

We also check that no chunk is accidentally assigned to two segment IDs.


In [ ]:
duplicate_assignment = (
    chunk_df.groupby("chunk_id")["segment_index"]
    .nunique()
    .gt(1)
    .sum()
)

print("Chunks assigned to multiple segments:", duplicate_assignment)
print("Chunks with missing document:", chunk_df["document"].isna().sum())

fund_counts = (
    chunk_df.groupby(["document", "fund"], dropna=False)
    .size()
    .reset_index(name="chunks")
)

display(fund_counts)


## 6. Inspect a known fund transition

Around page 38, ownership should change from
`Enhanced Intelligence Global Allocation Responsible Investing`
to `K&H egészség`.

No chunk should contain content from both funds.


In [ ]:
boundary_38 = chunk_df[
    (chunk_df["document"] == "FU_BF065_EN.pdf")
    & (chunk_df["page_end"] >= 36)
    & (chunk_df["page_start"] <= 40)
][
    [
        "chunk_id",
        "fund",
        "segment_type",
        "section",
        "page_start",
        "page_end",
        "word_count",
    ]
]

display(boundary_38)


## 7. Inspect the annex region

The final prospectus region should remain fund-aware:
- pages 500–514 → Enhanced Intelligence...
- pages 515–528 → World Selection 100-1 Advanced
- pages 529–542 → ČSOB Akciové...
- pages 543 onward → ČSOB Globálních...


In [ ]:
annex_chunks = chunk_df[
    (chunk_df["document"] == "FU_BF065_EN.pdf")
    & (chunk_df["page_end"] >= 500)
][
    [
        "chunk_id",
        "fund",
        "segment_type",
        "section",
        "page_start",
        "page_end",
        "word_count",
    ]
]

display(annex_chunks)


## 8. Inspect actual chunk text

This is the most important qualitative check: chunks should read naturally and should not start/end in obviously bad places.


In [ ]:
sample_columns = [
    "chunk_id",
    "fund",
    "segment_type",
    "section",
    "page_start",
    "page_end",
    "word_count",
    "text",
]

display(
    chunk_df[
        chunk_df["document"] == "FU_BF065_EN.pdf"
    ][sample_columns].head(8)
)


## 9. Save chunks after validation

These are generated artifacts and do not need to be committed to the interview repo.


In [ ]:
CHUNK_DIR = ROOT / "artifacts" / "chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_PATH = CHUNK_DIR / "chunks.json"
CHUNK_PATH.write_text(
    json.dumps(chunks, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Saved:", CHUNK_PATH)


## Conclusion

The last validated run produced **1,654 chunks** with a mean size of about **201 words**, median **222**, 95th percentile **300**, and **0 chunks above the 300-word maximum**. No chunk was assigned to multiple segments, and inspected transition/annex regions retained the correct fund ownership.

The current defaults (`max_words=300`, `min_words=80`, `overlap_words=40`) trade some redundancy for local continuity. A larger production corpus would likely tune these values against retrieval metrics rather than document aesthetics alone.
